# LOG05 | 미션 06 | Solution — 교관 전용

**학생에게 배포하지 마세요.** 이 Notebook은 근거 확인형 교관 해설, 모의 기록의 검토 예시와 평가 기준을 담습니다. 공개된 원자료가 없는 항목은 판정 불가로 남깁니다.

**미션:** 지휘소용 무인 보급장비 실시간 관제 및 긴급 중지 지침서 서식 개발

## 📦 실습자료 업로드 (모든 미션 공통 · 실습 전 필수)

아래 **「LOG05 실습자료 ZIP 업로드 및 압축 해제」** 코드셀을 **먼저 실행**하세요.

1. 교육기관에서 제공한 **`LOG05_실습프로젝트.zip`**을 준비합니다. 파일명이 다르더라도 LOG05 자료가 담긴 정상적인 ZIP 파일이면 업로드할 수 있습니다.
2. **Google Colab:** 셀 실행 후 나타나는 업로드 창에서 ZIP을 선택하면 자동으로 압축을 해제합니다. **로컬 Jupyter:** ZIP 선택 후 **「선택한 ZIP 압축 해제」** 버튼을 누릅니다.
3. 화면에서 **「압축 해제 완료」** 메시지와 실습자료 폴더 경로를 확인한 다음, 아래 미션 실습을 진행하세요.

**주의:** 업로드가 끝나지 않았다면 미션 실습을 시작하지 마세요. ZIP을 올렸다는 사실만으로 실습 결과나 교관 승인이 완료되는 것은 아닙니다. 아래 코드셀의 소스코드는 기본 숨김으로 설정되어 있습니다.


In [ ]:
# @title 📦 LOG05 실습자료 ZIP 업로드 및 압축 해제 { display-mode: "form" }
# 학습자 화면에는 결과와 업로드 인터페이스만 표시하도록 구성합니다.
from pathlib import Path
import os, zipfile, stat
from IPython.display import display, Markdown

LOG05_PROJECT_DIR = Path.cwd() / "LOG05_실습자료"
LOG05_PROJECT_READY = False

def _log05_extract_zip(zip_path):
    global LOG05_PROJECT_READY
    zip_path = Path(zip_path)
    if not zipfile.is_zipfile(zip_path):
        raise ValueError("유효한 ZIP 압축파일이 아닙니다. 실습자료 ZIP을 다시 선택해 주세요.")
    dst = LOG05_PROJECT_DIR.resolve()
    dst.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        entries = archive.infolist()
        if len(entries) > 20000 or sum(info.file_size for info in entries) > 1024**3:
            raise ValueError("압축파일의 항목 수 또는 해제 후 용량이 허용 범위를 초과합니다.")
        for info in entries:
            item = Path(info.filename.replace('\\', '/'))
            if item.is_absolute() or '..' in item.parts or not info.filename.strip():
                raise ValueError("안전하지 않은 ZIP 내부 경로가 포함되어 있습니다: " + info.filename)
            target = (dst / item).resolve()
            if not target.is_relative_to(dst):
                raise ValueError("실습 폴더 밖으로 벗어나는 파일 경로입니다.")
            mode = (info.external_attr >> 16) & 0o170000
            if mode == stat.S_IFLNK:
                raise ValueError("심볼릭 링크가 포함된 ZIP은 허용하지 않습니다.")
        for info in entries:
            target = dst / Path(info.filename.replace('\\', '/'))
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as src, target.open('wb') as out:
                    import shutil
                    shutil.copyfileobj(src, out)
    files = [p for p in dst.rglob('*') if p.is_file()]
    if not files:
        raise ValueError("ZIP에 압축 해제할 실습자료 파일이 없습니다.")
    LOG05_PROJECT_READY = True
    print("✅ LOG05 실습자료 압축 해제 완료:", dst)
    print("📄 확인된 파일 수:", len(files))
    print("ℹ️ 실습자료 폴더:", dst)
    print("※ 업로드·압축 해제는 학습 수행 및 평가 완료를 뜻하지 않습니다.")
    return dst

display(Markdown("**실습 시작 전 필수:** 교육기관에서 제공한 `LOG05_실습프로젝트.zip`을 업로드하세요. 이름이 다르더라도 정상적인 `.zip` 파일을 선택할 수 있지만, **LOG05 실습자료가 포함된 ZIP**이어야 합니다. 업로드 및 압축 해제를 완료한 후 다음 실습 단계로 진행하세요."))
try:
    from google.colab import files as _colab_files
    print("📤 업로드 창에서 LOG05 실습자료 ZIP 파일을 선택하세요.")
    _uploaded = _colab_files.upload()
    _zip_files = [(name, content) for name, content in _uploaded.items() if name.lower().endswith('.zip')]
    if not _zip_files:
        print("⚠️ ZIP 파일이 업로드되지 않았습니다. 이 셀을 다시 실행하세요.")
    else:
        if len(_zip_files) > 1:
            print("ℹ️ 여러 ZIP 중 첫 번째 파일을 사용합니다:", _zip_files[0][0])
        _name, _data = _zip_files[0]
        _zip_path = Path.cwd() / Path(_name).name
        _zip_path.write_bytes(_data)
        _log05_extract_zip(_zip_path)
except ImportError:
    import ipywidgets as _widgets
    _file_input = _widgets.FileUpload(accept='.zip', multiple=False, description='ZIP 파일 선택')
    _extract_button = _widgets.Button(description='선택한 ZIP 압축 해제', button_style='success')
    _upload_message = _widgets.Output()
    def _get_uploaded_zip():
        value = _file_input.value
        if not value:
            raise ValueError('먼저 LOG05 실습자료 ZIP을 선택하세요.')
        item = next(iter(value.values())) if isinstance(value, dict) else value[0]
        name = item['name']
        data = item['content']
        if not name.lower().endswith('.zip'):
            raise ValueError('ZIP 파일만 선택할 수 있습니다.')
        return name, bytes(data)
    def _on_extract(_):
        with _upload_message:
            _upload_message.clear_output()
            try:
                name, data = _get_uploaded_zip()
                target = Path.cwd() / Path(name).name
                target.write_bytes(data)
                _log05_extract_zip(target)
            except Exception as exc:
                print('⚠️ 업로드 또는 압축 해제 실패:', exc)
    _extract_button.on_click(_on_extract)
    display(_widgets.VBox([_file_input, _extract_button, _upload_message]))
    print('로컬 Jupyter: ZIP을 선택한 뒤 「선택한 ZIP 압축 해제」 버튼을 누르세요.')


## 교관 해설 및 판단 예시

관제 메모 M-1～M-3을 사실·판단·조치·미확인으로 구분합니다. 명령 발신, 수신 확인, 실제 이행 기록은 별개입니다. 실제 통신·제어 절차를 구현하거나 실습하지 않습니다.

**예시 풀이의 기본 논리**

1. 원본 가상 자료에서 해당 사례·기록 번호, 기간과 분모를 찾습니다.
2. 관찰된 **사실**, AI가 제안한 **해석**, 학생이 내리는 **판단**, 부족한 **자료**를 별도로 적습니다.
3. 근거를 확인할 수 없는 주장에는 승인이나 성공 판정을 붙이지 않습니다.
4. 최초 답안과 수정 답안을 구분하며 AI 요청·답변 원문을 확인합니다.
5. 학생의 독립 수행과 실제 사람 평가 조건을 별도로 확인합니다.

**실제 제3자 평가/승인:** 제공된 문서만으로 자동 생성할 수 없습니다.

## 원자료에 근거한 교관용 작성 예시

### 예시 결과: 가상 관제 메모를 네 갈래로 구분

| 구분 | 작성 예시 |
|---|---|
| 사실 | 원자료에 실제 기재된 시각·보고 내용·발신 기록 |
| 판단 | 해당 기록을 보고 학습자가 내린 해석과 근거 |
| 조치 | 교육용 문서에 기록한 검토·보고 이력(실제 장비 명령 아님) |
| 미확인 | 수신 확인 또는 실제 이행 여부가 보고되지 않은 상태 |

**핵심 오류 예시:** '메시지를 보냈다'를 '장비가 명령대로 움직였다'로 기록하면 근거 없는 단정입니다. 원문에 수신 및 이행 기록이 따로 있는지 확인해야 합니다.

관제 메모 3건의 근거를 동일한 방식으로 분류하며 실제 장비 통신 절차나 제어 코드는 만들지 않습니다.

## 학생과 동일한 원본 자료

미션 6. 지휘소용 무인 보급장비 실시간 관제 및 긴급 중지 지침서 서식 개발

**가. 제공 자료 1 - 혼동 유발 표현 예시(나머지는 조별로 6개 이상으로 늘린다)**

| 혼동 유발 표현 | 왜 혼동되는가 | 대체 표현 예 |
| --- | --- | --- |
| 정상입니다 | 무엇이 정상인지 보이지 않음 | 배터리 (%), 링크 (등급), 구간 (  ), 확인 단계 G(  ) |
| 정지하라 | 제자리 안전 정지인지 임무 중지인지 불명 | 제자리 안전 정지(또는 복귀, 원격 회수 요청)를 명시 |
| 곧 복귀합니다 | 복귀 시각과 조건이 없음 | 복귀 개시 (시각), 예상 도착 (시각) |

**가. 제공 자료 2 - 가상 관제 메모 3종(사실·판단·조치·미확인을 구분해 표시한다)**

| 메모 | 내용 |
| --- | --- |
| M-1 | UV-2 링크가 B 구간에서 3등급으로 떨어졌다. 곧 두절될 것 같다. 속도를 줄이라고 지시했다. 두절 시 정지 규칙을 조종수가 알고 있는지는 모른다. |
| M-2 | UD-1 배터리가 31%로 표시된다. 복귀하면 충분하다고 판단한다. 복귀를 지시했다. 관제 화면 값과 기체 표시가 다를 수 있다. |
| M-3 | C 구간 풍속이 3등급이라는 보고가 왔다. 임무를 계속해도 된다고 본다. 다음 보고를 5분 뒤로 정했다. 측정 시각은 확인되지 않았다. |

**나. 작성 서식 1 - 관제 상황판(한 줄 한 임무)**

| 임무 | 장비 | 구간 | 배터리 | 링크 등급 | 상태(녹·황·적) | 조치 |
| --- | --- | --- | --- | --- | --- | --- |

**나. 작성 서식 2 - 긴급 중지 명령 전달 순서(빈칸을 채운다): 1 관제 책임자가 (   ) 용어로 지시한다. 2 원격 조종수가 (   )하고 시각을 말한다. 3 기록자가 (   )에 기록한다. 4 (   ) 확인이 없으면 (   )초 후 다시 지시한다. 5 승인권자에게 (   ) 이내 보고한다.**

**나. 작성 서식 3 - 요약형 기록(메모 M-1～M-3을 옮겨 적는다)**

| 메모 | 사실 | 판단 | 조치 | 미확인 |
| --- | --- | --- | --- | --- |
| M-1 |  |  |  |  |
| M-2 |  |  |  |  |
| M-3 |  |  |  |  |

**나. 작성 서식 4 - 상세형 기록(1건 선택)**

| 기록 번호 |  |
| --- | --- |

| 사실(시각, 장비, 구간, 수치, 확인 단계) |  |
| --- | --- |

| 판단(상태 등급과 근거) |  |
| --- | --- |

| 조치(누가, 언제까지) |  |
| --- | --- |

| 미확인(확인 방법, 책임 직책) |  |
| --- | --- |

**나. 작성 서식 5 - 혼동 유발 표현 금지 목록(6개 이상)**

| 번호 | 금지 표현 | 대체 표현 |
| --- | --- | --- |
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |
| 4 |  |  |
| 5 |  |  |
| 6 |  |  |

**다. 점검표**

| 점검 항목 | 확인 |
| --- | --- |
| 관제 상황판에 배터리, 링크 등급, 상태 등급, 조치 칸이 있다 | (   ) |
| 긴급 중지 명령의 지시, 복창, 기록, 재지시, 보고 순서를 정했다 | (   ) |
| 혼동 유발 표현 금지 목록을 6개 이상 만들고 대체 표현을 적었다 | (   ) |
| 메모 3종에서 사실과 판단을 혼동한 문장이 0건이다 | (   ) |
| 요약형과 상세형이 같은 내용을 담고 있다 | (   ) |

## 활동별 교관 검토 해설

### 활동 1. 메모의 네 종류 내용을 나눈다

**교관이 확인할 구체적인 수행:** 1. 단계 개요 목표: 앞선 미션(M1～M3)에서 수집 및 작성된 비정상 상황 메모(M-1～M-3)의 내용을 면밀히 분석하고, 이를 사실(Fact), 판단(Judgment), 지시(Instruction), 미확인(Unconfirmed)의 4가지 유형으로 명확히 분류하여 기록합니다. 수행 방식: 주관적인 느낌이나 섞여 있는 보고 내용을 객관적인 데이터(사실), 분석된 결론(판단), 하달된 행동 지침(지시), 그리고 정보가 누락되거나 불확실한 영역(미확인)으로 엄정하게 분리하여 구조화합니다. 2. 세부 수행 절차 M-1～M-3 메모 내용의 수집 및 해체: 이전 미션 과정에서 도출된 현장 관제 및 비상 상황 관련 메모 내용을 가져와 문장 단위로 해체하고 성격을 분석합니다. 4가지 정보 유형 분류: 사실(Fact): 센서 수치, 타임스탬프, 명확히 입증된 관측 로그 등 객관적인 데이터 항목. 판단(Judgment): 사실을 바탕으로 내린 장비 상태 진단, 위험도 평가, 상황 해석 결과. 지시(Instruction): 관제자나 승인권자가 현장 조종수 또는 장비에 내린 구체적인 행동 명령 및 통제 지침. 미확인(Unconfirmed): 통신 두절, 데이터 유실 등으로 인해 확인되지 않아 추가 조사가 필요한 정보 공백 영역. 유형별 표기 및 분류 결과 정리: 각 메모의 항목별로 4가지 분류 기호 또는 명칭을 직접 부착하고, 혼동되기 쉬운 판단과 사실의 경계를 명확히 구분하여 서식에 기록합니다. 3. 주의 및 검증 사항 추정이나 가설에 불과한 내용을 '사실'로 분류하거나, 명확한 근거 없이 미확인 영역을 임의로 해소된 것으로 처리하지 않도록 주의해야 합니다. 본 단계에서 분류된 4가지 유형 결과는 다음 단계인 

**예시 답안의 근거:** 학생이 원본 자료 번호와 해당 원문·분모·조건을 연결했는지 확인합니다. 입력 자료가 없는 값은 '확인 필요'로 남겨야 합니다.

**평가 포인트:** 이 활동의 확인 근거 M06 활동1 — 메모의 네 종류 내용을 나눈다: M-1～M-3을 사실·판단·지시·미확인으로 직접 표시한다. 확인 증거는 실제 제공 자료번호·본인 작성/질문·AI 회신·대조/수정의 해당 위치입니다. 계획만 작성했다면 실행 완료로 표시하지 않습니다. 단계 결과를 forms/LOG05_M06_forms.md와 최종 파일 templates/log05_aar_fact_action_template.md, LOG05_M06_forms.md, forms/LOG05_M06_forms.md에 연결합니다. 보완 방법 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다. 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 2. AI와 관제·보고 서식을 만든다

**교관이 확인할 구체적인 수행:** 1. 단계 개요 목표: 앞선 1단계에서 4가지 유형(사실, 판단, 지시, 미확인)으로 분류한 메모 내용을 바탕으로, AI 에이전트와 협업하여 실시간 관제 상황판, 지시-복창-기록-재지시-보고의 표준 절차 순서, 현장 통제 시 준수해야 할 금지 표현 6개, 그리고 요약/상세 보고 서식 을 체계적으로 설계합니다. 수행 방식: 단편적이거나 구두로 이루어지는 임의의 보고 체계를 지양하고, 비정상 상황 발생 시 지휘소와 현장 간 오해나 혼선을 원천 차단할 수 있는 표준화된 관제 및 보고 체계를 구조화합니다. 2. 세부 수행 절차 실시간 관제 상황판 설계: 장비의 위치, 배터리/유류 잔량, 통신 링크 상태, 경보 발생 여부 등을 직관적으로 파악하고 모니터링할 수 있는 상황판의 필수 구성 항목을 정의합니다. 지시-복창-기록-재지시-보고 표준 순서 확립: 관제자의 명령 하달부터 현장의 복창, 상황판 기록, 필요시 재지시, 최종 상급 보고로 이어지는 5단계 커뮤니케이션 흐름과 필수 점검 포인트를 구체화합니다. 금지 표현 6가지 선정 및 통제: 현장 통제 및 보고 과정에서 혼란을 초래할 수 있는 모호한 표현(예: "적당히 조치", "신속히 이동", "임의 판단", "상황 종료 추정" 등) 6가지를 도출하고 사용을 엄격히 제한합니다. 요약 및 상세 보고 서식 분리 작성: 긴급 상황 발생 시 핵심 정보를 즉시 전달하기 위한 '요약 보고' 서식과, 사후 분석 및 감사를 위한 '상세 보고' 서식을 각각 설계하여 상황에 맞게 적용하도록 정비합니다. 3. 주의 및 검증 사항 AI가 제안하는 관제 서식이나 보고 순서가 실제 현장의박진감 넘치는 통신 제약 환경이나 긴급 중지 상황에 실효성이 있는지 꼼꼼히 대조해야 합니다. 금지 

**예시 답안의 근거:** 학생이 원본 자료 번호와 해당 원문·분모·조건을 연결했는지 확인합니다. 입력 자료가 없는 값은 '확인 필요'로 남겨야 합니다.

**평가 포인트:** 이 활동의 확인 근거 M06 활동2 — AI와 관제·보고 서식을 만든다: 상황판·지시/복창/기록/재지시/보고 순서·금지 표현 6개·요약/상세를 작성한다. 확인 증거는 실제 제공 자료번호·본인 작성/질문·AI 회신·대조/수정의 해당 위치입니다. 계획만 작성했다면 실행 완료로 표시하지 않습니다. 단계 결과를 forms/LOG05_M06_forms.md와 최종 파일 templates/log05_aar_fact_action_template.md, LOG05_M06_forms.md, forms/LOG05_M06_forms.md에 연결합니다. 보완 방법 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다. 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 3. 원문과 완료 주장을 대조한다

**교관이 확인할 구체적인 수행:** 1. 단계 개요 목표: 앞서 분류한 비정상 상황 메모(M-1～M-3)의 원문 내용과, 현장 보고에서 주장하는 누락 사항, 시간 추정, 장비 복귀 가능 여부, 수신 완료 여부 등의 완료 주장을 일대일로 면밀히 대조하고 보완합니다. 수행 방식: 통신 두절 상황이나 불완전한 로그 기록을 바탕으로 성급하게 도출된 완료 주장(예: 수신 완료, 즉시 복귀 가능 등)이 실제 원문 데이터 및 제약 조건과 일치하는지 철저히 검증합니다. 2. 세부 수행 절차 메모 원문과 완료 주장 간 일대일 대조: M-1부터 M-3까지의 실제 기록 원문과 현장에서 올라온 완료 주장(처리 결과)을 비교하여, 논리적 비약이나 데이터 누락이 없는지 항목별로 점검합니다. 시간 추정의 타당성 검토: 통신 지연이나 음영 구간에서 소요 시간을 임의로 단축하여 추정한 부분이 있는지 확인하고, 실제 계측 가능한 시간 기준에 맞춰 바로잡습니다. 복귀 가능 여부 및 수신 완료 여부 재검증: 불확실한 통신 신호 상태임에도 장비의 복귀가 완료되었거나 메시지가 정상 수신되었다고 단정한 주장의 근거를 원자료와 대조하여 엄정하게 평가합니다. 미비점 보완 및 수정: 대조 과정에서 식별된 오류나 과장된 완료 주장을 수정하고, 객관적으로 입증된 사실과 미확인 영역을 분리하여 기록을 보완합니다. 3. 주의 및 검증 사항 현장의 조치 완료 보고나 AI의 요약 설명만 믿고 원문 데이터의 불일치나 누락을 간과하지 않도록 주의해야 합니다. 본 단계에서 검증·보완된 내용은 다음 단계인 표준 표현 및 상태 이관(4단계)과 최종 서식 작성( LOG05_M06_forms.md , templates/log05_aar_fact_action_template.md 등)의 핵심 근거 자료

**예시 답안의 근거:** 학생이 원본 자료 번호와 해당 원문·분모·조건을 연결했는지 확인합니다. 입력 자료가 없는 값은 '확인 필요'로 남겨야 합니다.

**평가 포인트:** 이 활동의 확인 근거 M06 활동3 — 원문과 완료 주장을 대조한다: 같은 3메모의 누락·시간 추정·복귀 가능·수신 완료를 검토하고 보완한다. 확인 증거는 실제 제공 자료번호·본인 작성/질문·AI 회신·대조/수정의 해당 위치입니다. 계획만 작성했다면 실행 완료로 표시하지 않습니다. 단계 결과를 forms/LOG05_M06_forms.md와 최종 파일 templates/log05_aar_fact_action_template.md, LOG05_M06_forms.md, forms/LOG05_M06_forms.md에 연결합니다. 보완 방법 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다. 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

### 활동 4. 표준 표현과 상태를 넘긴다

**교관이 확인할 구체적인 수행:** 1. 단계 개요 목표: 앞선 1～3단계를 통해 수립 및 검증한 실시간 관제 상황판 , 비정상 메모의 4가지 유형(사실·판단·지시·미확인) 변환 결과 , 상세 분석 3건 및 검증 완료된 1건의 내용 , 그리고 지시-복창-기록-재지시-보고 체계의 명령 기록 서식 을 후속 미션인 미션 7(통신 재밍 및 기상 급변 모의 긴급 임무 중지 판단 스트레스 훈련)과 미션 9(피드백 반영 체크리스트 항목 보완)로 정확하고 누락 없이 인계합니다. 수행 방식: 개별 미션의 성과물로 단절되지 않고, 도출된 관제 상황판 운영 기준과 표준 통신 표현 규칙들을 후속 실습 및 최종 지침서 편성에 온전히 반영할 수 있도록 정돈된 형태로 저장 및 이관합니다. 2. 세부 수행 절차 실시간 관제 상황판 및 변환 결과 정리: 장비 상태 모니터링을 위해 설계된 관제 상황판의 구성 요소와, 비정상 상황 메모(M-1～M-3)를 사실·판단·지시·미확인 유형으로 변환한 최종 데이터를 체계적으로 정리합니다. 상세 분석 내용 및 명령 기록 서식 이관: 원문 대조 과정을 거쳐 오류가 바로잡힌 상세 분석 결과와, 지휘소와 현장 간 오해를 방지하기 위해 확립된 '지시-복창-기록-재지시-보고' 순서 및 금지 표현 통제 지침을 문서화합니다. 필수 파일 경로 연계 및 저장: 정리된 내용을 LOG05_M06_forms.md , forms/LOG05_M06_forms.md , templates/log05_aar_fact_action_template.md 등 지정된 결과 파일 경로에 빠짐없이 반영하고 저장합니다. 후속 미션(M7 및 M9) 연계 지점 확정: 본 미션에서 확정된 관제 표준 표현과 상태 데이터가 미션 7의 비상 스트레스 훈련 시나리오 및 미션 9의 

**예시 답안의 근거:** 학생이 원본 자료 번호와 해당 원문·분모·조건을 연결했는지 확인합니다. 입력 자료가 없는 값은 '확인 필요'로 남겨야 합니다.

**평가 포인트:** 이 활동의 확인 근거 M06 활동4 — 표준 표현과 상태를 넘긴다: 상황판·3건 변환·상세 1건·명령 기록 서식을M7·M9에 전달한다. 확인 증거는 실제 제공 자료번호·본인 작성/질문·AI 회신·대조/수정의 해당 위치입니다. 계획만 작성했다면 실행 완료로 표시하지 않습니다. 단계 결과를 forms/LOG05_M06_forms.md와 최종 파일 templates/log05_aar_fact_action_template.md, LOG05_M06_forms.md, forms/LOG05_M06_forms.md에 연결합니다. 보완 방법 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다. 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

## 원본 CMS 평가 기준

**평가 항목 1:** [정량적 평가] [선택] 관제·긴급 중지 서식 — 기록 분류 규칙을 100% 표준화하고 혼동 유발 표현 금지 가이드를 6개 이상 포함했는가, 가상 관제 메모 3종에서 사실과 판단의 혼동이 0건인가, 요약형·상세형 듀얼 템플릿을 구성하고 긴급 중지 명령 전달·복창 순서를 정했는가. PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다. 목표미달·0기준·미측정은그대로표시합니다. 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.

**평가 항목 2:** [정성적 평가] Why/How/What: 본인 계획·실제 대화·근거 검토3건 이상·보완·재확인·최종 반영. PASS 증거 지휘소에서 무인 보급장비를 실시간으로 관제하고 긴급 중지 명령을 내릴 때 쓰는 관제 상황판과 중지 지침서 서식을 개발한다.의판단이제공자료·역할책임·교육용가정과연결됩니다. 사실/제안/미확인과실제/모의수행을구분합니다. 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.

**평가 항목 3:** [진행 추적 및 완료 기준] 실제 조원·두 평가자·2분 판단·20분 문서점검·3분 발표·실제 교관 확인을 AI 모의와 구분한다. 미실시를 PASS로 쓰지 않는다. 파일: templates/log05_aar_fact_action_template.md, LOG05_M06_forms.md, forms/LOG05_M06_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다. 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다. 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.

**교관 검토 기록:** 판정 [미확인 / 보완 필요 / PASS / FAIL]을 실제 기준에 따라 기입해야 하며, 자동 표시는 하지 않습니다.

In [ ]:
MISSION_JSON = '{"code": "LOG05_m06", "steps": [{"title": "메모의 네 종류 내용을 나눈다"}, {"title": "AI와 관제·보고 서식을 만든다"}, {"title": "원문과 완료 주장을 대조한다"}, {"title": "표준 표현과 상태를 넘긴다"}]}'
# LOG05 학습 기록 도구 (표준 Python/Jupyter 환경용)
import json, zipfile, io, os, datetime
from pathlib import Path
from IPython.display import display, Markdown
try:
    import ipywidgets as widgets
except ImportError as exc:
    raise RuntimeError("ipywidgets가 필요합니다. 폐쇄망에서는 사전 설치된 로컬 Jupyter 또는 제공된 Markdown 양식을 사용하세요.") from exc

MISSION_DATA = json.loads(MISSION_JSON)
CODE = MISSION_DATA['code']
OUT_DIR = Path(CODE + '_results')
OUT_DIR.mkdir(exist_ok=True)
BACKUP = OUT_DIR / (CODE + '_record.json')
fields = {}
for no, item in enumerate(MISSION_DATA['steps'], 1):
    fields['activity_%02d' % no] = widgets.Textarea(
        value='', placeholder='내 판단, 확인한 자료 번호, AI 제안과 수정 이유를 직접 작성하세요.',
        description='활동 %s' % no, layout=widgets.Layout(width='98%',height='130px'),
        style={'description_width':'75px'})
for key, label in [('first_thought','AI에게 묻기 전 내 판단'), ('ai_request','실제로 보낸 AI 요청 전체'),
                   ('ai_response','실제로 받은 AI 답변 전체'), ('source_check','원자료 대조 3건과 수정 이유'),
                   ('final_result','최종 작성 결과와 근거'), ('limits','미실시/미측정/확인 필요 및 교관 검토 상태')]:
    fields[key] = widgets.Textarea(value='', description=label,
       layout=widgets.Layout(width='98%',height='125px'),style={'description_width':'190px'})
status = widgets.HTML(value='<b>아직 저장하지 않았습니다.</b> 작성과 교관 승인은 별개입니다.')
restore_input = widgets.Textarea(description='복원 JSON',placeholder='백업 JSON의 내용을 붙여넣으세요.',layout=widgets.Layout(width='98%',height='100px'),style={'description_width':'110px'})

def payload():
    return {'schema':'log05-student-v1','mission':CODE,'saved_at':datetime.datetime.now(datetime.timezone.utc).isoformat(),
            'answers':{key:w.value for key,w in fields.items()},
            'status':'작성 기록 (학습 PASS 또는 교관 승인 아님)'}

def save_record(_=None):
    obj=payload()
    if BACKUP.exists():
        stamp=datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
        BACKUP.rename(OUT_DIR / (CODE+'_previous_'+stamp+'.json'))
    BACKUP.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf8')
    status.value='<b>로컬 저장 완료:</b> '+str(BACKUP)+' (PASS 아님)'

def export_json(_=None):
    save_record()
    raw=BACKUP.read_bytes()
    try:
        from google.colab import files
        files.download(str(BACKUP))
    except ImportError:
        print('Colab이 아닙니다. 저장 위치:',BACKUP.resolve())

def restore_record(_=None):
    try:
        obj=json.loads(restore_input.value)
        if obj.get('mission')!=CODE or not isinstance(obj.get('answers'),dict):raise ValueError('다른 미션 또는 형식이 맞지 않는 백업입니다.')
        save_record()  # 복원 전 백업 저장
        for key,w in fields.items():w.value=str(obj['answers'].get(key,''))
        status.value='<b>복원 적용:</b> 기존 작성 내용은 이전 JSON으로 보관했습니다. 다시 저장하세요.'
    except Exception as exc:status.value='<b>복원 오류:</b> '+str(exc)

def export_review_zip(_=None):
    save_record()
    target=OUT_DIR / (CODE+'_review.zip')
    with zipfile.ZipFile(target,'w',compression=zipfile.ZIP_DEFLATED) as zz:
        zz.write(BACKUP,arcname=BACKUP.name)
        text='\n\n'.join('## '+key+'\n'+val for key,val in payload()['answers'].items())
        zz.writestr(CODE+'_work.md',text)
        zz.writestr('README.txt','검토용 보관 파일. 실제 평가 PASS 또는 교관 승인 자료가 아닙니다.')
    try:
        from google.colab import files
        files.download(str(target))
    except ImportError:
        print('Colab이 아닙니다. ZIP 위치:',target.resolve())

for i,item in enumerate(MISSION_DATA['steps'],1):
    display(Markdown('### 활동 %d. %s' % (i,item['title'])))
    display(fields['activity_%02d' % i])
for key,label in [('first_thought','최초 판단'),('ai_request','AI 요청 원문'),('ai_response','AI 답변 원문'),
 ('source_check','원자료 대조'),('final_result','최종 결과'),('limits','한계 및 실제 검토 상태')]:
    display(Markdown('### '+label));display(fields[key])
buttons=[]
for title,handler in [('작성 내용 저장',save_record),('작성 JSON 내려받기',export_json),('검토용 ZIP 내려받기',export_review_zip),('작성 JSON 복원',restore_record)]:
    b=widgets.Button(description=title,button_style='info');b.on_click(handler);buttons.append(b)
display(widgets.HBox(buttons[:2]));display(widgets.HBox(buttons[2:]));display(restore_input);display(status)
